# 04y -- Composite Dynasty ADP (cross-source post-pass)

**Purpose:** Blend the per-source ADPs (`ktc_adp` crowd sentiment + `ds_adp`
projection model) into a single source-agnostic **`composite_adp`**, plus a
**`sources_count`** confidence companion. The two ADPs measure different
philosophies on incommensurable scales (KTC = overall-pick integers; DS =
round.pick), so they are **percentile-normalized within (source, format)** before
averaging — never raw-averaged.

**Runs AFTER** `04b` (KTC) and `04x` (DynastySharks/FantasyPros) have written
their partitions. Writes a new `source_name = "Composite"` partition to
`fact_dynasty_ranking_metrics`, keyed by `gsis_id` (cross-source rows have no
backbone row, so they relate to players via gsis only). Run with CWD = repo root.

In [1]:
# ---- Setup ------------------------------------------------------------------
from pathlib import Path
import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG

FACT        = f"{CFG.data_dir}/fact_dynasty_ranking_metrics.parquet"
SOURCE_NAME = "Composite"
ADP_KEYS    = ["ktc_adp", "ds_adp"]      # per-source ADPs to blend (extend if sources grow)
print("composite source:", SOURCE_NAME, "| blending:", ADP_KEYS)

composite source: Composite | blending: ['ktc_adp', 'ds_adp']


In [2]:
# ---- Blend: percentile within (source, format) -> mean -> re-rank -----------
m = pd.read_parquet(FACT)
adp = m[m["metric_key"].isin(ADP_KEYS) & m["gsis_id"].notna()
        & (m["metric_num"] > 0)].copy()   # 0 = upstream "no ADP" sentinel, not best

# Percentile within (source, format): each source's scale is normalized independently,
# so KTC's overall-pick integers and DS's round.pick encoding become comparable. pct
# ranks ascending -> lower adp (better) maps to a lower percentile.
adp["pctile"] = (adp.groupby(["source_name", "format"])["metric_num"]
                    .rank(method="average", pct=True))

# One adp per (source, format) per player; average the available percentiles across
# sources. Single-source players keep their sole percentile (sources_count flags this).
grp = (adp.groupby(["snapshot_date", "format", "gsis_id"])
          .agg(mean_pctile=("pctile", "mean"),
               sources_count=("source_name", "nunique"))
          .reset_index())

# Re-rank the blended percentile to a clean 1..N integer per (snapshot, format).
grp["composite_adp"] = (grp.groupby(["snapshot_date", "format"])["mean_pctile"]
                           .rank(method="min").astype(int))
print(f"composite players: {grp['gsis_id'].nunique()} | "
      f"both-source: {int((grp['sources_count'] == 2).sum())} | "
      f"single-source: {int((grp['sources_count'] == 1).sum())}")

composite players: 453 | both-source: 500 | single-source: 406


In [3]:
# ---- Emit long rows (composite_adp + sources_count) under 'Composite' --------
# Keyed by gsis_id; no source_player_id / source_uid (composite is
# cross-source -> relates to players via the gsis relationship, not to any backbone row).
recs = []
for r in grp.itertuples():
    base = {"snapshot_date": r.snapshot_date, "source_name": SOURCE_NAME,
            "source_player_id": None, "format": r.format, "source_uid": None,
            "gsis_id": r.gsis_id, "metric_text": None}
    recs.append({**base, "metric_key": "composite_adp", "metric_num": float(r.composite_adp)})
    recs.append({**base, "metric_key": "sources_count", "metric_num": float(r.sources_count)})

comp = pd.DataFrame.from_records(recs)
mn = etl.load_replace_partition(comp, FACT)         # replace the (snapshot, Composite) slice
print(f"[ok] wrote {len(comp)} Composite rows (table now {mn})")

[ok] wrote 1812 Composite rows (table now 26008)


In [4]:
# ---- Validation -------------------------------------------------------------
chk = pd.read_parquet(FACT)
comp_chk = chk[chk["source_name"] == SOURCE_NAME]
assert comp_chk["gsis_id"].notna().all(), "every composite row must carry gsis_id"

# each metric_key now maps to exactly one source_name (the FD that lets source live in the dim)
multi = (chk.groupby("metric_key")["source_name"].nunique().loc[lambda s: s > 1])
print("metric_keys with >1 source (should be empty):", multi.to_dict())

ca = comp_chk[comp_chk["metric_key"] == "composite_adp"]
print("composite_adp rows per format:")
print(ca.groupby("format").size().to_string())
print("\ntop 10 composite (SF):")
top = (ca[ca["format"] == "SF"].sort_values("metric_num").head(10)
       .merge(chk[chk["metric_key"] == "ktc_adp"][["gsis_id", "format", "metric_num"]]
                 .rename(columns={"metric_num": "ktc_adp"}), on=["gsis_id", "format"], how="left")
       .merge(chk[chk["metric_key"] == "ds_adp"][["gsis_id", "format", "metric_num"]]
                 .rename(columns={"metric_num": "ds_adp"}), on=["gsis_id", "format"], how="left"))
print(top[["gsis_id", "metric_num", "ktc_adp", "ds_adp"]]
      .rename(columns={"metric_num": "composite_adp"}).to_string(index=False))

metric_keys with >1 source (should be empty): {}
composite_adp rows per format:
format
SF      453
TEPP    453

top 10 composite (SF):
   gsis_id  composite_adp  ktc_adp  ds_adp
00-0040122            1.0      1.0    2.05
 LOV121782            2.0      1.0    2.06
00-0040666            3.0      3.0    2.09
00-0040124            4.0      5.0    3.04
00-0040691            5.0     13.0    2.11
00-0040126            6.0     11.0    3.05
 MEN516487            7.0      4.0    4.03
00-0040128            8.0     10.0    4.02
00-0040734            9.0      6.0    4.06
00-0040129           10.0     12.0    4.01
